# Summary

Test the Google RAG components

### Use the ADK environment


In [1]:
import os, sys

from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool
import vertexai

utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])


## Explore existing Corpora (note these are saved in GCP)

In [2]:
# See what methods are available in the rag object
# dir(rag)

# Find existing corpora and delete
existing_corpora = rag.list_corpora()
# print(existing_corpora)

# Get the corpus names and put in a list
existing_corpora_list = []
for existing_corpus in existing_corpora:
    existing_corpora_list.append(existing_corpus.name)
print(existing_corpora_list)

# Delete these
# for corpus_name in existing_corpora_list:
#     rag.delete_corpus(corpus_name)

# Check that there is nothing there
# for existing_corpus in existing_corpora:
#     existing_corpora_list.append(existing_corpus.name)


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/4752423506782715904', 'projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/7274439298110193664', 'projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/2518638091606949888']


## Get the contents of the bucket that has text files

In [3]:
# Get a list of contents in the GCP bucket that will be used for corpus
gcs_bucket_name = "ccc-crawl_data_xb"
# gcs_path = "crawl_data/text_files"
gcs_path = "crawl_data/jsonl_files"
bcontents = gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                gcs_bucket_name=gcs_bucket_name,
                                path=gcs_path)
print(bcontents)


['crawl_data/jsonl_files/', 'crawl_data/jsonl_files/calmattersdigitaldemocracyorg_2025May01_text.jsonl', 'crawl_data/jsonl_files/cccaoeorg_2025May01_text.jsonl', 'crawl_data/jsonl_files/ccrctccolumbiaedu_2025May01_text.jsonl', 'crawl_data/jsonl_files/enwikipediaorg_2025May01_text.jsonl', 'crawl_data/jsonl_files/icangotocollegecom_2025May01_text.jsonl', 'crawl_data/jsonl_files/laocagov_2025May01_text.jsonl', 'crawl_data/jsonl_files/nscresearchcenterorg_2025May01_text.jsonl', 'crawl_data/jsonl_files/wwwaaccncheedu_2025May01_text.jsonl', 'crawl_data/jsonl_files/wwwccccoedu_2025May01_text.jsonl', 'crawl_data/jsonl_files/wwwccleagueorg_2025May01_text.jsonl', 'crawl_data/jsonl_files/wwwecsorg_2025May01_text.jsonl']


## Create and save a corpus

In [4]:
# Create an embedding model
embedding_model_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model="publishers/google/models/text-embedding-005"
    )
)


In [6]:
# Create a corpus
display_name = "ccc_test_corpus"
rag_corpus = rag.create_corpus(
    display_name=display_name,
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_model_config
    ),
)


In [7]:
# Import Files to the RagCorpus
paths = ["gs://{}/{}".format(gcs_bucket_name, gcs_path)]
rag.import_files(
    rag_corpus.name,
    paths,
    # Optional
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512,
            chunk_overlap=100,
        ),
    ),
    max_embedding_requests_per_min=600,  # Optional
)


imported_rag_files_count: 11

## If needed look for the file(s) that failed

In [8]:
# find the file that failed
corpus_files = rag.list_files(rag_corpus.name)

# Get the displaynmes
corpus_file_display_names = [f.display_name for f in corpus_files]

# Check what's missing
bcontents_files = [os.path.split(bc)[1] for bc in bcontents]
missing_files = [f for f in bcontents_files if f not in corpus_file_display_names]
missing_files


['']

## Create a RAG Corpus using a Vertex AI engine

In [2]:
# Name your corpus
display_name = "ccc_test_corpus_vai"

# Vertex AI Search name
ENGINE_NAME = os.environ["ENGINE_NAME"]

vertex_ai_search_config = rag.VertexAiSearchConfig(
    serving_config=f"{ENGINE_NAME}/servingConfigs/default_search",
)

rag_corpus = rag.create_corpus(
    display_name=display_name,
    vertex_ai_search_config=vertex_ai_search_config,
)


## Create a RAG Corpus using Vertex AI

In [2]:
# Name your corpus
display_name = "ccc_test_corpus_vai"

# Vertex AI Search name
ENGINE_NAME = os.environ["ENGINE_NAME"]

vertex_ai_search_config = rag.VertexAiSearchConfig(
    # serving_config=f"{ENGINE_NAME}/servingConfigs/default_search",
    serving_config=f"{ENGINE_NAME}"
)

rag_corpus = rag.create_corpus(
    display_name=display_name,
    vertex_ai_search_config=vertex_ai_search_config,
)

## Get the corpus name

In [3]:
# Check that the new corpus is there
existing_corpora = rag.list_corpora()
existing_corpora_list = []
for existing_corpus in existing_corpora:
    existing_corpora_list.append(existing_corpus.name)
    existing_corpora_list.append(existing_corpus.update_time)

existing_corpora_list


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/4752423506782715904',
 DatetimeWithNanoseconds(2025, 6, 10, 16, 41, 0, 610735, tzinfo=datetime.timezone.utc),
 'projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/7274439298110193664',
 DatetimeWithNanoseconds(2025, 6, 14, 13, 50, 52, 256085, tzinfo=datetime.timezone.utc),
 'projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/2518638091606949888',
 DatetimeWithNanoseconds(2025, 6, 16, 10, 56, 45, 207172, tzinfo=datetime.timezone.utc),
 'projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/5112711476972355584',
 DatetimeWithNanoseconds(2025, 6, 17, 1, 53, 26, 486508, tzinfo=datetime.timezone.utc)]

In [5]:
dir(existing_corpus)

['__bool__',
 '__class__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'corpus_status',
 'create_time',
 'description',
 'display_name',
 'name',
 'update_time',
 'vector_db_config',
 'vertex_ai_search_config']